# Tema 5 · Relacionar texto e imagen y explorar conceptos

Seis actividades sincronizadas con el PDF. Ruta principal: CPU y entorno básico del curso, sin descargar un modelo grande ni usar claves de API. Si abres desde Colab, la primera celda descarga los recursos del repositorio (unos pocos MB); en local usa los archivos descargados con el repositorio.

Los vectores incluidos proceden de **inferencia real de CLIP**: se guardaron junto a la revisión del modelo y las huellas de sus imágenes. No son vectores inventados. Puedes recombinar las consultas incluidas; para frases nuevas, utiliza el notebook opcional `05B_clip_en_vivo.ipynb`.

Los dos soportes de móvil fueron generados previamente con `image_gen`; los prompts exactos están en los recursos. No generamos imágenes con un modelo dentro de este notebook. El experimento de ruido de la actividad 5 solo ilustra un proceso directo, sin aprender a invertirlo.

In [ ]:
from pathlib import Path
import json, hashlib
from urllib.request import urlopen
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

BASE_URL = "https://raw.githubusercontent.com/afalco/vision-ia-producto/main/05_multimodal_generativa/recursos/"
opciones = [Path("recursos"), Path("05_multimodal_generativa/recursos"),
            Path("vision-ia-producto/05_multimodal_generativa/recursos")]
RECURSOS = next((p for p in opciones if (p/"catalogo.json").exists()), Path("recursos_tema5"))
RECURSOS.mkdir(parents=True, exist_ok=True)
def recurso(nombre):
    destino = RECURSOS / nombre
    if not destino.exists():
        print("Descargando recurso:", nombre)
        try:
            with urlopen(BASE_URL + nombre, timeout=60) as respuesta:
                destino.write_bytes(respuesta.read())
        except Exception as error:
            raise RuntimeError("No se pudo descargar " + nombre +
                ". Comprueba que los cambios del tema 5 estén publicados o descarga el repositorio completo.") from error
    return destino

catalogo = json.loads(recurso("catalogo.json").read_text(encoding="utf-8"))
procedencia = json.loads(recurso("procedencia_clip.json").read_text(encoding="utf-8"))
for nombre, esperado in procedencia["sha256_archivos"].items():
    actual = hashlib.sha256(recurso(nombre).read_bytes()).hexdigest()
    if actual != esperado:
        raise ValueError("Recurso cambiado o incompleto: " + nombre + ". Descarga una versión coherente del tema.")
consultas = catalogo["consultas"]
nombres = [e["id"] for e in catalogo["imagenes"]]
imagenes = [Image.open(recurso(e["archivo"])).convert("RGB") for e in catalogo["imagenes"]]
with np.load(recurso("clip_vectores.npz"), allow_pickle=False) as datos:
    vectores_imagen = datos["imagenes"].copy()
    vectores_texto = datos["textos"].copy()
    escala_logits = float(datos["escala"])
assert vectores_imagen.shape == (len(imagenes), procedencia["vector_dim"])
assert vectores_texto.shape == (len(consultas), procedencia["vector_dim"])
assert np.allclose(np.linalg.norm(vectores_imagen, axis=1), 1, atol=1e-5)
assert np.allclose(np.linalg.norm(vectores_texto, axis=1), 1, atol=1e-5)
print("Modelo:", procedencia["modelo"], "Revisión:", procedencia["revision"])
print("Galería:", len(imagenes), "imágenes; banco:", len(consultas), "consultas")

## Actividad 1 · Leer el catálogo (10 min)

Explora la galería. Las seis primeras imágenes son ilustraciones procedimentales del curso; A y B son renders generados con IA. Los identificadores no se enviaron al modelo: CLIP recibió píxeles y textos. No hay una taza verde ni una escena con dos tazas azules.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, imagen, nombre in zip(axes.ravel(), imagenes, nombres):
    ax.imshow(imagen)
    ax.set_title(nombre)
    ax.axis("off")
plt.tight_layout()
plt.show()
print("Consultas disponibles:")
for i, frase in enumerate(consultas): print(i, frase)

**Predice:** ¿qué debería devolver una búsqueda de una taza verde si no existe? ¿Puede un ranking devolver siempre algo aunque la respuesta correcta sea «ninguno»?

**Tu respuesta:** …

## Actividad 2 · Buscar con texto (12 min)

CLIP representa imágenes y textos en un espacio compartido. Sus vectores están normalizados: el producto escalar coincide con el coseno. La puntuación no es probabilidad ni una comprobación de requisitos. En esta ruta solo se admiten las frases del banco; el notebook opcional permite escribir otras.

In [ ]:
similitudes = vectores_imagen @ vectores_texto.T

def buscar(frase, k=3):
    if frase not in consultas:
        raise ValueError("Frase no incluida en el banco. Elige una consulta disponible o usa 05B_clip_en_vivo.ipynb.")
    j = consultas.index(frase)
    ids = np.argsort(-similitudes[:, j])[:k]
    return pd.DataFrame({"Imagen": [nombres[i] for i in ids],
                         "Coseno": similitudes[ids, j]})

CONSULTA = "a drawing of a blue mug"
print(buscar(CONSULTA).round(3).to_string(index=False))
j = consultas.index(CONSULTA)
ids = np.argsort(-similitudes[:, j])[:3]
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, i in zip(axes, ids):
    ax.imshow(imagenes[i])
    ax.set_title(f"{nombres[i]} · {similitudes[i, j]:.3f}")
    ax.axis("off")
plt.tight_layout()
plt.show()

**Modifica:** prueba la lámpara, la arandela y el soporte de móvil. Registra una predicción y el resultado. Los valores no son comparables con un umbral universal de aceptación; este catálogo no constituye una evaluación industrial.

**Tu respuesta:** …

## Actividad 3 · Clasificación sin entrenar una cabeza (12 min)

Comparamos la imagen de la taza azul con descripciones candidatas. La operación softmax reparte un total de 1 entre **las candidatas presentes** usando los logits escalados de CLIP. Puede asignar mucho peso a una opción incorrecta cuando falta la descripción adecuada; no es confianza calibrada de cumplimiento.

In [ ]:
def pesos_candidatas(nombre_imagen, frases):
    i = nombres.index(nombre_imagen)
    js = [consultas.index(f) for f in frases]
    logits = escala_logits * similitudes[i, js]
    pesos = np.exp(logits - logits.max())
    pesos /= pesos.sum()
    return pd.DataFrame({"Candidata": frases, "Coseno": similitudes[i, js],
                         "Peso relativo softmax": pesos})

completas = ["a drawing of a blue mug", "a drawing of a red mug", "a green mug"]
incompletas = ["a drawing of a red mug", "a green mug"]
print("Con descripción adecuada:")
print(pesos_candidatas("taza_azul", completas).round(3).to_string(index=False))
print("Sin descripción adecuada:")
print(pesos_candidatas("taza_azul", incompletas).round(3).to_string(index=False))

**Comprueba:** la suma es 1 en ambos casos. ¿Significa que alguna candidata tiene que ser correcta? ¿Qué cambia: el vector de la imagen, su coseno con cada frase o los pesos normalizados? No hemos entrenado un clasificador nuevo: reutilizamos el preentrenamiento.

**Tu respuesta:** …

## Actividad 4 · Idioma, negación y cantidad (12 min)

Compara las variantes. La tarjeta de este modelo indica que su uso se limita al inglés; una coincidencia puntual en castellano no demuestra capacidad multilingüe validada. Negación, conteo y propiedades finas necesitan comprobación explícita.

In [ ]:
variantes = ["a drawing of a blue mug", "a photo of a blue mug",
             "un dibujo de una taza azul", "a blue cup with a handle",
             "a blue mug without a handle", "two blue mugs", "a green mug"]
filas = []
for frase in variantes:
    j = consultas.index(frase)
    i = int(similitudes[:, j].argmax())
    filas.append({"Consulta": frase, "Primera imagen": nombres[i],
                  "Coseno": float(similitudes[i, j])})
comparacion = pd.DataFrame(filas)
print(comparacion.round(3).to_string(index=False))
comparacion.to_csv("tema5_busquedas.csv", index=False)

**Analiza:** examina las consultas de color ausente, sin asa y dos tazas. Distingue parecido general y satisfacción literal. Un resultado alto no certifica número, geometría ni fabricabilidad. Elige dos fallos y propón cómo los comprobarías con datos o una herramienta específica.

**Tu respuesta:** …

## Actividad 5 · Añadir ruido no es generar (12 min)

Ilustramos un proceso directo: combinamos una imagen conocida con ruido. El coeficiente alpha es la fracción de señal en esta fórmula, no una instrucción universal de un generador. En difusión, el proceso inverso requiere un modelo aprendido y un procedimiento de muestreo. **Aquí no se aprende ni se ejecuta esa inversión.**

In [ ]:
x0 = np.asarray(imagenes[nombres.index("concepto_B")].resize((192, 192)), dtype=np.float32)/127.5 - 1
rng = np.random.default_rng(42)
epsilon = rng.normal(size=x0.shape)
alphas = [1.0, 0.7, 0.2, 0.0]
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, alpha in zip(axes, alphas):
    xt = np.sqrt(alpha)*x0 + np.sqrt(1-alpha)*epsilon
    # Solo limitamos el rango para mostrar, no como operación del proceso aprendido.
    ax.imshow(np.clip((xt+1)/2, 0, 1))
    ax.set_title(f"alpha = {alpha}")
    ax.axis("off")
plt.tight_layout()
plt.show()

**Predice y modifica:** cambia la semilla. ¿Qué permanece en alpha=1 y qué cambia en alpha=0? ¿Por qué quitar exactamente este ruido conociendo `x0` no demostraría que un modelo aprendió a crear imágenes nuevas?

**Tu respuesta:** …

## Actividad 6 · Del prompt al concepto revisable (14 min)

Compara A (instrucción abierta) y B (instrucción con restricciones visuales). Son **dos generaciones independientes, sin semilla controlada**: no permiten atribuir causalmente todas las diferencias al prompt. Ambas son conceptos visuales, no CAD ni piezas listas para imprimir. El modelo generador exacto y la semilla no fueron expuestos por la herramienta; no atribuimos una arquitectura específica a estas imágenes.

In [ ]:
prompts = json.loads(recurso("prompts_generacion.json").read_text(encoding="utf-8"))
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, letra in zip(axes, ["A", "B"]):
    ax.imshow(imagenes[nombres.index("concepto_"+letra)])
    ax.set_title("Concepto " + letra)
    ax.axis("off")
plt.tight_layout()
plt.show()
for e in prompts["conceptos"]:
    print("\n", e["archivo"], "\n", e["prompt"])
revision = pd.DataFrame([
    {"Requisito": "Dos apoyos frontales y hueco central para cable", "A": "Por revisar", "B": "Por revisar", "Verificación adicional": "Inspección visual y ajuste con dimensiones del cable"},
    {"Requisito": "Conectividad de un único cuerpo", "A": "Por revisar", "B": "Por revisar", "Verificación adicional": "Modelo CAD o malla cerrada revisada"},
    {"Requisito": "Espesores mínimos suficientes", "A": "No verificable solo con imagen", "B": "No verificable solo con imagen", "Verificación adicional": "Dimensiones y proceso definidos"},
    {"Requisito": "Estabilidad con móvil y carga", "A": "No verificable solo con imagen", "B": "No verificable solo con imagen", "Verificación adicional": "Geometría, centro de masas y ensayo"},
    {"Requisito": "Orientación y necesidad de soportes", "A": "Por revisar", "B": "Por revisar", "Verificación adicional": "CAD, laminador, material y equipo"}
])
print(revision.to_string(index=False))
revision.to_csv("tema5_revision_conceptos.csv", index=False)

**Completa:** sustituye «Por revisar» por observaciones justificadas. Puedes mantener «No verificable» cuando falte evidencia. Redacta una nueva instrucción cambiando una sola familia de requisitos y propón cómo compararías varias muestras de forma más controlada. No hace falta contratar un servicio de generación para entregar.

**Tu respuesta y prompt revisado:** …

## Entrega

Incluye tres búsquedas comentadas, dos fallos de interpretación literal, explicación del experimento de ruido y tabla de revisión con un prompt mejorado. Descarga tus CSV antes de cerrar Colab. No presentes un render como geometría fabricable.

## Fuentes

- [Tarjeta del modelo CLIP](https://huggingface.co/openai/clip-vit-base-patch32).
- [Documentación de CLIP en Transformers](https://huggingface.co/docs/transformers/model_doc/clip).
- [Rombach et al.: Latent Diffusion Models](https://arxiv.org/abs/2112.10752).

Consulta `recursos/procedencia_clip.json` y `recursos/prompts_generacion.json` para identificar el origen de los recursos.